In [2]:
read_directory_historical = "/bdd/CORDEX/output/EUR-11/GERICS/CNRM-CERFACS-CNRM-CM5/historical/r1i1p1/GERICS-REMO2015/v2/day/tas/latest/"
#read_directory_rcp = "/home/user/These/cordex_htws_cc3d/Data/rcp85"
read_directory_rcp = "/bdd/CORDEX/output/EUR-11/GERICS/CNRM-CERFACS-CNRM-CM5/rcp85/r1i1p1/GERICS-REMO2015/v2/day/tas/latest/"
path_to_remove = "/bdd/CORDEX/output/EUR-11/"
#write_directory = join("/data/tmandonnet/CORDEX/test/output",read_directory_rcp.replace(path_to_remove,""))

In [8]:
import numpy as np
import pandas as pd
import os
from os import listdir
from os.path import isfile, join
import re
import xarray as xr
from tqdm import tqdm
import errno
from detection_overlap_functions import *
from tqdm import tqdm #create a user-friendly feedback while script is running
import cc3d #connected components patterns
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as clrs
from scipy import stats
from scipy import signal
import json

In [ ]:
with open('succeded_combination.txt') as f:
    lines = f.readlines()
combination = ('NCC-NorESM1-M', 'CLMcom-ETH-COSMO-crCLIM-v1-1', 'rcp85', 'v1')
flag = False
count=0
while flag==False and count<len(lines):
    if str(combination) in lines[count] :
        flag=True
    count+=1

print(flag)

False


In [2]:
f1 = "CORDEX_EUR-11_land_area_fraction_Lambert_Conformal.nc"
f2 = "CORDEX_EUR-11_land_area_fraction_rotated_latitude_longitude.nc"
f3 = "CORDEX_EUR-11_land_area_fraction_rotated_pole.nc"

In [99]:
directory = "/home/user/These/cordex_htws_cc3d/Data/sftlf"

d1 = xr.open_dataset(join(directory,f1),engine='netcdf4')
d2 = xr.open_dataset(join(directory,f2),engine='netcdf4')
d3 = xr.open_dataset(join(directory,f3),engine='netcdf4')

In [9]:
read_directory_historical = "/home/user/These/cordex_htws_cc3d/Data/historical"
read_directory_rcp = "/home/user/These/cordex_htws_cc3d/Data/rcp85"
write_directory = "/home/user/These/cordex_htws_cc3d/Data/"
start_year = 1991
end_year = 2005
start_year_ref = 1951
end_year_ref = 2020
interval = 5
temp_variable = 'tas'
smooth_span = 15
threshold_value = 95
distrib_window_size = 15
anomaly = True
relative_threshold = True
nb_days = 4
split_year=2005

In [6]:
read_directory_historical = "/home/user/These/cordex_htws_cc3d/Data/historical"
read_directory_rcp = "/home/user/These/cordex_htws_cc3d/Data/rcp85"
write_directory = "/home/user/These/cordex_htws_cc3d/Data/"
start_year=1971
end_year=2020
start_year_ref=1976
end_year_ref=1995
interval=5
temp_variable='tas'
smooth_span=15
split_year=2005

In [ ]:
read_directory_historical=read_directory_historical
read_directory_rcp=read_directory_rcp
write_directory=write_directory
start_year=1971
end_year=2100

start_year_ref=1976
end_year_ref=2005
ref_period_offset=0.61

interval=5
running_mean_window_size=20
regional_warming_levels_list=[2.1,2.6,4.0,5.1]

In [20]:
if running_mean_window_size % 2 != 0:
    raise ValueError(f"running_mean_window_size must be an even integer, found {running_mean_window_size}")

# Create list of years of the beginning of each file
year_list = range(start_year,end_year,interval)
print(year_list)
#Create list of files to load
correct_files_list=[""]*len(year_list)
for i in range(len(year_list)) :
    year = year_list[i]
    if year<=split_year : # Before split_year, historical run
        correct_files_list[i] = join(read_directory_historical,find_correct_year_file(read_directory_historical,year_list[i],interval))
    else : # After split_year, RCP (4.5 or 8.5)
        correct_files_list[i] = join(read_directory_rcp,find_correct_year_file(read_directory_rcp,year_list[i],interval))

print(correct_files_list)

range(1971, 2100, 5)
['/home/user/These/cordex_htws_cc3d/Data/historical/tas_EUR-11_CNRM-CERFACS-CNRM-CM5_historical_r1i1p1_GERICS-REMO2015_v2_day_19710101-19751231.nc', '/home/user/These/cordex_htws_cc3d/Data/historical/tas_EUR-11_CNRM-CERFACS-CNRM-CM5_historical_r1i1p1_GERICS-REMO2015_v2_day_19760101-19801231.nc', '/home/user/These/cordex_htws_cc3d/Data/historical/tas_EUR-11_CNRM-CERFACS-CNRM-CM5_historical_r1i1p1_GERICS-REMO2015_v2_day_19810101-19851231.nc', '/home/user/These/cordex_htws_cc3d/Data/historical/tas_EUR-11_CNRM-CERFACS-CNRM-CM5_historical_r1i1p1_GERICS-REMO2015_v2_day_19860101-19901231.nc', '/home/user/These/cordex_htws_cc3d/Data/historical/tas_EUR-11_CNRM-CERFACS-CNRM-CM5_historical_r1i1p1_GERICS-REMO2015_v2_day_19910101-19951231.nc', '/home/user/These/cordex_htws_cc3d/Data/historical/tas_EUR-11_CNRM-CERFACS-CNRM-CM5_historical_r1i1p1_GERICS-REMO2015_v2_day_19960101-20001231.nc', '/home/user/These/cordex_htws_cc3d/Data/historical/tas_EUR-11_CNRM-CERFACS-CNRM-CM5_histor

In [10]:
# Create list with all files needed to compute climatology
onlyfiles = [f for f in listdir(read_directory_historical) if isfile(join(read_directory_historical, f))] #+ [f for f in listdir(read_directory_rcp) if isfile(join(read_directory_rcp, f))]
# Create list of years of the beginning of each file
year_list = range(start_year,end_year,interval)

In [7]:
year_list

range(2021, 2100, 5)

In [12]:
correct_files_list=[]
for year in year_list :
    if year<=2005 : # Before 2005, historical run
        correct_files_list.append(join(read_directory_historical,find_correct_year_file(read_directory_historical,year,interval))) 
    else : # After 2005, RCP (4.5 or 8.5)
        correct_files_list.append(join(read_directory_rcp,find_correct_year_file(read_directory_rcp,year,interval))) 


In [13]:
correct_files_list

['/home/user/These/cordex_htws_cc3d/Data/historical/tas_EUR-11_CNRM-CERFACS-CNRM-CM5_historical_r1i1p1_GERICS-REMO2015_v2_day_19710101-19751231.nc',
 '/home/user/These/cordex_htws_cc3d/Data/historical/tas_EUR-11_CNRM-CERFACS-CNRM-CM5_historical_r1i1p1_GERICS-REMO2015_v2_day_19760101-19801231.nc',
 '/home/user/These/cordex_htws_cc3d/Data/historical/tas_EUR-11_CNRM-CERFACS-CNRM-CM5_historical_r1i1p1_GERICS-REMO2015_v2_day_19810101-19851231.nc',
 '/home/user/These/cordex_htws_cc3d/Data/historical/tas_EUR-11_CNRM-CERFACS-CNRM-CM5_historical_r1i1p1_GERICS-REMO2015_v2_day_19860101-19901231.nc',
 '/home/user/These/cordex_htws_cc3d/Data/historical/tas_EUR-11_CNRM-CERFACS-CNRM-CM5_historical_r1i1p1_GERICS-REMO2015_v2_day_19910101-19951231.nc',
 '/home/user/These/cordex_htws_cc3d/Data/historical/tas_EUR-11_CNRM-CERFACS-CNRM-CM5_historical_r1i1p1_GERICS-REMO2015_v2_day_19960101-20001231.nc']

In [3]:
ds = xr.open_dataset('/home/user/These/cordex_htws_cc3d/Data/rcp85/tas_EUR-11_CNRM-CERFACS-CNRM-CM5_rcp85_r1i1p1_GERICS-REMO2015_v2_day_20210101-20251231.nc',engine='netcdf4')

In [19]:
ds = xr.open_mfdataset(correct_files_list, engine='netcdf4')


In [9]:
compute_regional_warming_levels(read_directory_historical,read_directory_rcp,write_directory,start_year=start_year,end_year=end_year,start_year_ref=start_year_ref,end_year_ref=end_year_ref,regional_warming_levels_list=[2.1,2.6,4.0,10])

{2.1: None, 2.6: None, 4.0: None, 10: None}

In [ ]:
def compute_regional_warming_levels(read_directory_historical,read_directory_rcp,write_directory,start_year=1971,end_year=2100,start_year_ref=1986,end_year_ref=2005,ref_period_offset=0.72,interval=5,running_mean_window_size=20,regional_warming_levels_list=[2.1,2.6,4.0,5.1]) :
    # RWL values gathered from https://interactive-atlas.ipcc.ch/regional-information by selecting the four reference regions overlapping with EUR CORDEX domain, and taking the median temperature of each GWL period in the Table Summary
    # Default RWL 0.72°C of the 1986-2005 period is computed in compute_regional_offset.ipynb
    if running_mean_window_size % 2 != 0:
        raise ValueError(f"running_mean_window_size must be an even integer, found {running_mean_window_size}")

    # Create list of years of the beginning of each file
    year_list = range(start_year,end_year,interval)
    #Create list of files to load
    correct_files_list=[""]*len(year_list)
    for i in range(len(year_list)) :
        year = year_list[i]
        if year<=split_year : # Before split_year, historical run
            correct_files_list[i] = join(read_directory_historical,find_correct_year_file(read_directory_historical,year_list[i],interval))
        else : # After split_year, RCP (4.5 or 8.5)
            correct_files_list[i] = join(read_directory_rcp,find_correct_year_file(read_directory_rcp,year_list[i],interval))

    # Load multi-file dataset
    ds = xr.open_mfdataset(correct_files_list, engine='netcdf4', chunks={'time': 1461})
    # Variable is necessarily 'tas' for the computation of warming level
    da = ds.tas
    da = da.groupby(da.time.dt.year).mean() # Compute annual mean

    # Create reference period (default 1986-2005) average
    mask = (da.year >= start_year_ref)&(da.year <= end_year_ref)
    da_ref = da.sel(year=mask)
    da_ref = da_ref.mean(dim="year") # Average over time, 1 time step remaining

    da = da.rolling(year=running_mean_window_size, center=True).mean() # Compute running mean
    # Compute latitude-weighted mean to obtain 1D-array of annual mean
    weights = np.cos(np.deg2rad(da.lat))
    weights.name = "weights"
    da_weighted = da.weighted(weights)
    da_ref = da_ref.weighted(weights)
    try : # Have to take into account the fact that grid_mapping is not always the same in CORDEX outputs
        da_weighted = da_weighted.mean(("rlat","rlon")) # 1D-array of annual values
        da_ref = da_ref.mean(("rlat","rlon")) # one single value
    except :
        da_weighted = da_weighted.mean(("x","y")) # 1D-array of annual values
        da_ref = da_ref.mean(("x","y")) # one single value
        
    da_weighted = da_weighted - da_ref # 1D-array of annual values reprenting the european warming since the reference period (default 1986-2005)
    warming_levels_dict = {}
    for level in regional_warming_levels_list :
        level_diff = level - ref_period_offset # Since reference period is not 1850-1900, have to introduce offset (default is 0.61°C for 1976-2005 reference period)
        # Find years warmer than level and get the first matching year
        bool_array = (da_weighted - level_diff > 0)
        if not bool_array.any() : # If the corresponding level has not been reached by the model
            warming_levels_dict[level] = None
        else :
            central_year = int(bool_array.idxmax())
            warming_levels_dict[level] = {"start_year" : int(central_year - running_mean_window_size/2), "end_year" : int(central_year + (running_mean_window_size/2 - 1))}
    
    #with open(join(write_directory,'regional_warming_levels.json'), 'w') as fp:
    #    json.dump(warming_levels_dict, fp)

    return warming_levels_dict

In [ ]:
RWL_dict = {'2.1': {'start_year': 2036, 'end_year': 2055}, '2.6': {'start_year': 2046, 'end_year': 2065}, '4.0': {'start_year': 2073, 'end_year': 2092}, '5.1': None}

In [ ]:
# Account for the fact that RWL can overlap. Create 4 RWL columns to avoid edge cases but RWL_3 and RWL_4 should remain empty
RWL_col_dict = {1:"RWL_1",2:"RWL_2",3:"RWL_3",4:"RWL_4"}
# Create list of computed years (only the years that match a RWL period)
RWL_years = []
for k in RWL_dict.keys() :
    if RWL_dict[k] is not None :
        RWL_years += list(range(RWL_dict[k]["start_year"],RWL_dict[k]["end_year"]))
RWL_years = np.unique(RWL_years)
df_heatwaves = pd.DataFrame(data=None,columns=["Year","RWL_1","RWL_2","RWL_3","RWL_4","Mean","Max","Duration","Spatial Extent","Temp sum"])
for year in RWL_years :
    label = "" # Load the corresponding nc file
    for label in np.unique(label.data) : # Iterate over the heatwaves (one label = one heatwave)
        df_heatwaves.loc[label,"Year"]=Year
        # Find the corresponding RWL(s)
        rwl_count=0
        for rwl in RWL_dict :
            if year in range(RWL_dict[rwl]["start_year"],RWL_dict[rwl]["end_year"]) :
                rwl_count += 1 # Find the correct column to fill
                df_heatwaves.loc[label,RWL_col_dict[rwl_count]] = float(rwl)
            

In [84]:
float('2.1')

2.1

In [80]:
rwl = 2.1
df_heatwaves[(df_heatwaves["RWL_1"]==rwl) | (df_heatwaves["RWL_2"]==rwl) | (df_heatwaves["RWL_3"]==rwl) | (df_heatwaves["RWL_4"]==rwl)]

,Year,RWL_1,RWL_2,RWL_3,RWL_4
0,2010,2.1,2.6,4.0,None
1,2015,2.1,2.6,None,None
